## Init session

In [ ]:
%load_ext autoreload
%autoreload 2

### Install dependencies

In [ ]:
!chmod +x install.sh
! ./install.sh > /dev/null 2>&1

### Import packages

In [1]:
import os
import boto3
import subprocess

from pathlib import Path
from random import randint

from rich.pretty import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import torch


from sklearn.model_selection import train_test_split

import model as ic

## Download from S3 bucket

In [ ]:
# 1. Automatically retrieve credentials from the Onyxia environment
mybucket = "jbarrere"
key = os.environ.get('AWS_ACCESS_KEY_ID')
secret = os.environ.get('AWS_SECRET_ACCESS_KEY')
token = os.environ.get('AWS_SESSION_TOKEN')
endpoint = os.environ.get('AWS_ENDPOINT_URL')
print(key)

# 2. Initialize the client
s3 = boto3.client("s3",endpoint_url = endpoint,
                  aws_access_key_id = key, 
                  aws_secret_access_key = secret, 
                  aws_session_token = token)

# 3. Check connection
listobjbucket = s3.list_objects_v2(Bucket=mybucket, MaxKeys=5)
if 'Contents' in listobjbucket:
    print("Connection OK!\nFiles:")
    for obj in listobjbucket['Contents']:
        print("-", obj['Key'])

In [ ]:
# Download data.zip if needed
if not os.path.exists("/home/onyxia/work/data.zip"):
    s3.download_file(mybucket, "data.zip", "data.zip")

# Download mlflow if needed
if not os.path.exists("/home/onyxia/work/mlflow.db"):
    s3.download_file(mybucket, "mlflow.db", "mlflow.db")

# Download mlruns if needed
if not os.path.exists("/home/onyxia/work/mlruns.zip"):
    s3.download_file(mybucket, "mlruns.zip", "mlruns.zip")

# Unzip data.zip if needed
if not os.path.exists("/home/onyxia/work/data"):
    subprocess.run(["unzip", "data.zip"],
                   stdout=subprocess.DEVNULL,
                   stderr=subprocess.DEVNULL)
    print("Done!")

## Load & clean data

### Import annotations

In [2]:
binary_columns_todrop = ["animal", "no", "water", "zoom"]
binary_columns = ["human", "anthropic", "vegetation", "rock", "snow"]
tab_raw = pd.read_csv(Path(".").joinpath("data").joinpath("annotations.csv")).drop(
    binary_columns_todrop, axis = 1
)
print(tab_raw)

            image         site  human  anthropic  vegetation  rock  snow
0     10057685324  Carpathians      0          1           1     0     0
1     10057694024  Carpathians      0          0           0     1     0
2     10057713565  Carpathians      0          0           1     1     0
3     10057821923  Carpathians      0          0           1     0     0
4     10067284414  Carpathians      0          0           1     0     0
...           ...          ...    ...        ...         ...   ...   ...
5432   9697490796    Vinschgau      1          1           1     1     0
5433   9739872961    Vinschgau      0          1           1     0     0
5434   9739878607    Vinschgau      0          1           1     0     0
5435   9739881163    Vinschgau      0          0           1     0     1
5436   9835114964    Vinschgau      0          1           1     0     0

[5437 rows x 7 columns]


### Select sites

In [3]:
sites_trainvaltest = ["Carpathians", "French_Alps", "Stubai_Valley", "Vinschgau"]
sites_external = ["Danube", "Dovre", "Sierra_Nevada"]

tab_external = tab_raw[tab_raw["site"].isin(sites_external)].copy()
tab_raw = tab_raw[tab_raw["site"].isin(sites_trainvaltest)]

### Rebalance categories

In [4]:
n = 2000
rng = np.random.default_rng(42)           # ------------------------------------------------------------------ #
# Compute weights from imbalance in the original dataframe
# ------------------------------------------------------------------ #
# p = proportion of 1s; distance from 0.5 ranges from 0 (perfect balance)
# to 0.5 (all 0s or all 1s). We map it to a weight >= 1.
# weight = 1 + k * (|p - 0.5| / 0.5)  with k controlling the max weight.
k = 4  # max additional weight on top of the baseline 1
weights = {}
for col in binary_columns:
    p = tab_raw[col].mean()
    imbalance = abs(p - 0.5) / 0.5  # 0 = perfectly balanced, 1 = fully skewed
    weights[col] = 1 + k * imbalance
    
# ------------------------------------------------------------------ #
# Greedy balanced selection
# ------------------------------------------------------------------ #
seed = 42  
rng = np.random.default_rng(seed)
df_shuffled = tab_raw.sample(frac=1, random_state=seed).reset_index(drop=True)

selected_indices = []
counts = {col: {0: 0, 1: 0} for col in binary_columns}
target = n // 2

base_tolerance = 500   
ramp = 1000

values = df_shuffled[binary_columns].values

for idx, row_vals in enumerate(values):
    if len(selected_indices) >= n:
        break

    score = 0

    for j, col in enumerate(binary_columns):
        val = int(row_vals[j])
        w = weights[col]

        score += w * (
            (target - counts[col][val])
            - (target - counts[col][1 - val])
        )

    # -------- trade-off control --------
    progress = len(selected_indices) / n
    threshold = -(base_tolerance + progress * ramp)

    if score >= threshold:
        selected_indices.append(idx)

        for j, col in enumerate(binary_columns):
            counts[col][int(row_vals[j])] += 1

In [5]:
# Make two entry datasets : one balanced and one with all data
tab_balanced = df_shuffled.loc[selected_indices].reset_index(drop = True)
tab_full = df_shuffled.copy()

In [6]:
# Show how categories are balanced (or not)
# - Balanced dataframe
pd.DataFrame(
    data={
        "Category": [col for col in binary_columns],
        "%": [tab_balanced[col].mean() * 100 for col in binary_columns],
        "Number": [sum(tab_balanced[col]) for col in binary_columns],
        "Total": len(tab_balanced),
        "Data": "balanced"
    }
).sort_values("%", ascending = False)

,Category,%,Number,Total,Data
2,vegetation,61.869536,920,1487,balanced
1,anthropic,50.302623,748,1487,balanced
0,human,49.764627,740,1487,balanced
3,rock,49.697377,739,1487,balanced
4,snow,49.630128,738,1487,balanced


In [7]:
# Full dataframe
pd.DataFrame(
    data={
        "Category": [col for col in binary_columns],
        "%": [tab_full[col].mean() * 100 for col in binary_columns],
        "Number": [sum(tab_full[col]) for col in binary_columns],
        "Total": len(tab_full),
        "Data": "full"
    }
).sort_values("%", ascending = False)

,Category,%,Number,Total,Data
2,vegetation,80.681431,2368,2935,full
1,anthropic,47.427598,1392,2935,full
3,rock,46.371380,1361,2935,full
0,human,29.369676,862,2935,full
4,snow,27.086882,795,2935,full


## Minigrid experiment

In [8]:
def run_minigrid(tab, exp_name):
    """
    Run the complete experiment pipeline with stratification and hyperparameter tuning.
    
    Parameters:
    -----------
    tab : pd.DataFrame
        The input dataframe containing the data
    exp_name : str
        The name of the experiment (used for logging)
    """
    # Stratify dataset
    tab_strat = tab.copy()
    tab_strat["strat"] = ""
    for col in tab.columns[2:]:
        tab_strat["strat"] += tab_strat[col].astype(str)

    # Split in train and validation dataset
    trainval, test = train_test_split(tab_strat, test_size=0.15, random_state=42, stratify=tab_strat["strat"])
    test = test.drop("strat", axis=1)

    # Set the backbones, repetitions, learning rates and batch size to experiment
    backbones = ["hf_swt_t", "hf_resnet", "hf_cnx2_t", "hf_vit_g16"]
    repetitions = [3, 4, 5] 
    learning_rates = [0.0001, 0.00001]
    batch_sizes = [16, 32]

    # Loop on all parameters
    for rep in repetitions:
        train, val = train_test_split(
            trainval,
            test_size=0.18,
            stratify=trainval["strat"],
            random_state=rep
        )
        train = train.drop("strat", axis=1)
        val = val.drop("strat", axis=1)
                
        for backbone in backbones:
            for lr in learning_rates:
                for bs in batch_sizes:
                        
                    ic.train_model(
                        train_data=train,
                        val_data=val,
                        batch_size=bs,
                        max_epochs=20,  
                        image_size=224,
                        run_owner="onyxia",
                        exp_name=exp_name,
                        backbone=backbone,
                        loss_name="bce",
                        loss_params={"alpha":0.25, "gamma":2},  
                        device=ic.get_device(),
                        checkpoints_n_saved=1,
                        learning_rate=lr,
                        early_stoper_patience=5,
                        early_stoper_min_delta=0.001,
                        use_lr_finder=False,
                        lr_scheduler_step=10,
                        lr_scheduler_gamma=0.85,
                        print_steps="print",
                        log_progress=False,
                        plot_loss=False,
                        num_workers=10,
                    )

In [ ]:
# Run minigrid experiment with balanced dataset
run_minigrid(tab=tab_balanced, exp_name="minigrid_balanced")

In [ ]:
# Run minigrid experiment with full dataset
run_minigrid(tab=tab_full, exp_name="minigrid_full")

### Export minigrid outputs

In [ ]:
t_runs = mlflow.search_runs(search_all_experiments=True, experiment_names=["minigrid_balanced"])
t_runs_df = t_runs[[ "run_id", "start_time", "params.batch_size", "params.learning_rate" ]].sort_values("start_time").reset_index(drop=True).copy()
k=0
for rep in [3, 4, 5]:
    for backbone in ["hf_swt_t", "hf_resnet", "hf_cnx2_t", "hf_vit_g16"]:
        for lr in [0.0001, 0.00001]:
            for bs in [16, 32]:
                if k < len(t_runs_df):
                    t_runs_df.loc[k, 'loop_rep'] = rep
                    t_runs_df.loc[k, 'loop_backbone'] = backbone
                    t_runs_df.loc[k, 'loop_lr'] = lr
                    t_runs_df.loc[k, 'loop_bs'] = bs
                k = k+1

print(t_runs_df)

In [9]:
def extract_mlflow_results_minigrid(experiment_names, output_file):
    """
    Extract and combine MLflow run results from specified experiments.
    
    Parameters:
    -----------
    experiment_names : list or str
        Name(s) of the MLflow experiment(s) to search
    output_file : str
        Path where the combined results CSV should be saved
    
    Returns:
    --------
    pd.DataFrame
        The combined results dataframe
    """
    # Convert single experiment name to list if needed
    if isinstance(experiment_names, str):
        experiment_names = [experiment_names]
    
    # Search for runs
    runs = mlflow.search_runs(search_all_experiments=True, experiment_names=experiment_names)

    ### - Dirty fix since backbone was not logged (to remove afterwards)
    runs_df = runs[[ "run_id", "start_time", "params.batch_size", "params.learning_rate" ]].sort_values("start_time").reset_index(drop=True).copy()
    k=0
    for rep in [3, 4, 5]:
        for backbone in ["hf_swt_t", "hf_resnet", "hf_cnx2_t", "hf_vit_g16"]:
            for lr in [0.0001, 0.00001]:
                for bs in [16, 32]:
                    runs_df.loc[k, 'backbone'] = backbone
                    k=k+1
    runs_df = runs_df.drop(columns="start_time")
    runs_df.columns = ["run_id", "batch_size", "learning_rate", "backbone"]
    ### - End of the dirty fix
    
    # Extract relevant parameters
    #runs_df = runs[[
    #    "run_id",
    #    "params.backbone",
    #    "params.batch_size",
    #    "params.learning_rate"
    #]].copy()
    #runs_df.columns = ["run_id", "backbone", "batch_size", "learning_rate"]
    
    # Collect all classification reports
    all_data = []
    for _, row in runs_df.iterrows():
        run_id = row["run_id"]
        
        try:
            local_path = mlflow.artifacts.download_artifacts(
                run_id=run_id,
                artifact_path="metrics/classification_report.csv"
            )
            
            df_report = pd.read_csv(local_path, sep=";")
            
            df_report["run_id"] = run_id
            df_report["backbone"] = row["backbone"]
            df_report["batch_size"] = row["batch_size"]
            df_report["learning_rate"] = row["learning_rate"]
            
            all_data.append(df_report)
            
        except Exception as e:
            print(f"Error for {run_id}: {e}")
    
    # Combine all results
    final_df = pd.concat(all_data, ignore_index=True)
    
    # Reorder columns
    cols_order = ["run_id", "backbone", "batch_size", "learning_rate"]
    final_df = final_df[
        cols_order + [col for col in final_df.columns if col not in cols_order]
    ]
    
    # Save to CSV
    final_df.to_csv(output_file, index=False)
    
    return final_df





In [10]:
# Extract output for the balanced minigrid
os.makedirs("outputs", exist_ok=True)
df_minigrid_balanced = extract_mlflow_results_minigrid(
    experiment_names=["minigrid_balanced"],
    output_file="outputs/results_minigrid_balanced.csv"
)
print(df_minigrid_balanced)

Error for 38dc216a38dc4bb9ac0e300440e7a675: Failed to download artifacts from path 'classification_report.csv', please ensure that the path is correct.


Error for 77d91226c53c496faf24bdb109506e60: Failed to download artifacts from path 'classification_report.csv', please ensure that the path is correct.


Error for 4a6f76de418f4c6795563cdd2d1d4a1b: Failed to download artifacts from path 'classification_report.csv', please ensure that the path is correct.


Error for cef6092622bf421282ec48f93561793c: Failed to download artifacts from path 'classification_report.csv', please ensure that the path is correct.


Error for 53d1e3144357447f85f6d8f11d4d44c7: Failed to download artifacts from path 'classification_report.csv', please ensure that the path is correct.


Error for 79d81518f086428a9f650448e5ce4465: Failed to download artifacts from path 'classification_report.csv', please ensure that the path is correct.
                               run_id    backbone batch_size learning_rate  \
0    eefa762140eb4635a59942b1cde72557    hf_swt_t         16        0.0001   
1    eefa762140eb4635a59942b1cde72557    hf_swt_t         16        0.0001   
2    eefa762140eb4635a59942b1cde72557    hf_swt_t         16        0.0001   
3    eefa762140eb4635a59942b1cde72557    hf_swt_t         16        0.0001   
4    eefa762140eb4635a59942b1cde72557    hf_swt_t         16        0.0001   
..                                ...         ...        ...           ...   
373  47d197599bda473888b0e8d31871c1b2  hf_vit_g16         32        0.0001   
374  47d197599bda473888b0e8d31871c1b2  hf_vit_g16         32        0.0001   
375  47d197599bda473888b0e8d31871c1b2  hf_vit_g16         32        0.0001   
376  47d197599bda473888b0e8d31871c1b2  hf_vit_g16         32        

In [11]:
# Extract output for the full minigrid
df_minigrid_full = extract_mlflow_results_minigrid(
    experiment_names=["minigrid_full"],
    output_file="outputs/results_minigrid_full.csv"
)
print(df_minigrid_full)

Error for 0fdd710871084743bfbb93ca65091117: Failed to download artifacts from path 'classification_report.csv', please ensure that the path is correct.


Error for f3e69da930444b069c558df39e21ac86: Failed to download artifacts from path 'classification_report.csv', please ensure that the path is correct.


Error for dc701f954571481d8513d1f70502d2e8: Failed to download artifacts from path 'classification_report.csv', please ensure that the path is correct.
                               run_id    backbone batch_size learning_rate  \
0    99c8f217233949ff95dbdd12214fce61    hf_swt_t         16        0.0001   
1    99c8f217233949ff95dbdd12214fce61    hf_swt_t         16        0.0001   
2    99c8f217233949ff95dbdd12214fce61    hf_swt_t         16        0.0001   
3    99c8f217233949ff95dbdd12214fce61    hf_swt_t         16        0.0001   
4    99c8f217233949ff95dbdd12214fce61    hf_swt_t         16        0.0001   
..                                ...         ...        ...           ...   
400  8d124331b7c540d3bafb2bb2f4f16dfc  hf_vit_g16         16         1e-05   
401  8d124331b7c540d3bafb2bb2f4f16dfc  hf_vit_g16         16         1e-05   
402  8d124331b7c540d3bafb2bb2f4f16dfc  hf_vit_g16         16         1e-05   
403  8d124331b7c540d3bafb2bb2f4f16dfc  hf_vit_g16         16        

### Train 50 models for hf_swt_t

In [ ]:
def train_models(tab, backbone, learning_rate, batch_size, exp_name):
    """
    Train a model with specified hyperparameters.
    
    Parameters:
    -----------
    tab : pd.DataFrame
        The input dataframe containing the data
    backbone : str
        The backbone model to use (e.g., "hf_swt_t", "hf_resnet", "hf_cnx2_t", "hf_vit_g16")
    learning_rate : float
        The learning rate for training (e.g., 0.0001, 0.00001)
    batch_size : int
        The batch size for training (e.g., 16, 32)
    exp_name : str
        The name of the experiment (used for logging)
    
    """
    # Stratify dataset
    tab_strat = tab.copy()
    tab_strat["strat"] = ""
    for col in tab.columns[2:]:
        tab_strat["strat"] += tab_strat[col].astype(str)

    # Split in train and validation dataset
    trainval, test = train_test_split(tab_strat, test_size=0.15, random_state=42, stratify=tab_strat["strat"])
    test = test.drop("strat", axis=1)

    # Loop over different random seeds for train/val splits
    for rep in [11, 12, 13, 14, 15]:
        train, val = train_test_split(
            trainval,
            test_size=0.18,
            stratify=trainval["strat"],
            random_state=rep
        )
        train = train.drop("strat", axis=1)
        val = val.drop("strat", axis=1)
        
        # Train multiple times with same split (10 repetitions)
        for _ in list(range(10)):
            ic.train_model(
                train_data=train,
                val_data=val,
                batch_size=batch_size,  
                max_epochs=100,
                image_size=224,
                run_owner="onyxia",
                exp_name=exp_name+str(rep),  
                backbone=backbone,  
                loss_name="bce",
                loss_params={"alpha": 0.25, "gamma": 2},
                device=ic.get_device(),
                checkpoints_n_saved=1,
                learning_rate=learning_rate,  
                early_stoper_patience=10,
                early_stoper_min_delta=0.001,
                use_lr_finder=False,
                lr_scheduler_step=10,
                lr_scheduler_gamma=0.85,
                print_steps="print",
                log_progress=False,
                plot_loss=False,
                num_workers=10,
            )

In [ ]:
# Train model with balanced dataset
train_models(
    tab=tab_balanced,
    backbone="hf_swt_t",
    learning_rate=0.0001,
    batch_size=32,
    exp_name="swin_balanced"
)

In [ ]:
# Train model with full dataset
train_models(
    tab=tab_full,
    backbone="hf_swt_t",
    learning_rate=0.0001,
    batch_size=32,
    exp_name="swin_full"
)

### Export trainswt outputs

In [ ]:
def export_mlflow_results_swin(exp_name, output_file, repetitions=None):
    """
    Export MLflow results from experiments where repetition is included in the experiment name.
    
    Parameters:
    -----------
    exp_name : str
        The base name of the experiment (e.g., "trainswt", "trainresnet")
    output_file : str
        Path where the combined results CSV should be saved
    repetitions : list, optional
        List of repetition numbers (default: [11, 12, 13, 14, 15])
    
    Returns:
    --------
    pd.DataFrame
        The combined results dataframe
    """
    if repetitions is None:
        repetitions = [11, 12, 13, 14, 15]
    
    all_data = []
    
    for rep in repetitions:
        # Get experiment by name with repetition number appended
        exp_name_full = f"{exp_name}{rep}"
        exp = mlflow.get_experiment_by_name(exp_name_full)
        
        if exp is None:
            print(f"Warning: Experiment '{exp_name_full}' not found. Skipping...")
            continue
        
        runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
        
        for _, row in runs.iterrows():
            run_id = row["run_id"]
            
            try:
                local_path = mlflow.artifacts.download_artifacts(
                    run_id=run_id,
                    artifact_path="metrics/classification_report.csv"
                )
                
                df_report = pd.read_csv(local_path, sep=";")
                
                df_report["run_id"] = run_id
                df_report["rep"] = rep
                
                all_data.append(df_report)
                
            except Exception as e:
                print(f"Error for {run_id}: {e}")
    
    if not all_data:
        print(f"No data found for experiments with base name: {exp_name}")
        return None
    
    final_df = pd.concat(all_data, ignore_index=True)
    
    cols_order = ["run_id", "rep"]
    final_df = final_df[
        cols_order + [col for col in final_df.columns if col not in cols_order]
    ]
    
    final_df.to_csv(output_file, index=False)
    print(f"Results saved to {output_file}")
    print(f"Total runs exported: {len(final_df)}")
    
    return final_df

In [ ]:
# Export results with balanced dataset
results_swin_balanced = export_mlflow_results_swin(
    exp_name="swin_balanced",
    output_file="outputs/swin_balanced_results.csv",
    repetitions=[11, 12, 13, 14, 15]
)
print(results_swin_balanced)

In [ ]:
# Export results with full dataset
results_swin_full = export_mlflow_results_swin(
    exp_name="swin_full",
    output_file="outputs/swin_full_results.csv",
    repetitions=[11, 12, 13, 14, 15]
)
print(results_swin_full)

### Find best model

In [ ]:
df_weighted = final_df[final_df["labels"] == "weighted avg"].copy()

print(df_weighted.loc[df_weighted["f1-score"].idxmax()])

In [ ]:
experiments = mlflow.search_experiments()
experiment_ids = [e.experiment_id for e in experiments if e.name.startswith("trainswt")]

runs = mlflow.search_runs(experiment_ids=experiment_ids)

best = runs.sort_values("params.F1_weighted_avg", ascending=False).iloc[0]

print(best)

best_run_id = best.run_id

### Export training outputs

In [ ]:
client = mlflow.tracking.MlflowClient()

train_loss = client.get_metric_history(best_run_id, "training Loss")
val_loss   = client.get_metric_history(best_run_id, "validation Loss")

train_f1 = client.get_metric_history(best_run_id, "training F1")
val_f1   = client.get_metric_history(best_run_id, "validation F1")


train_loss_df = pd.DataFrame([
    {"epoch": m.step, "train_loss": m.value}
    for m in train_loss
])

val_loss_df = pd.DataFrame([
    {"epoch": m.step, "val_loss": m.value}
    for m in val_loss
])

train_f1_df = pd.DataFrame([
    {"epoch": m.step, "train_f1": m.value}
    for m in train_f1
])

val_f1_df = pd.DataFrame([
    {"epoch": m.step, "val_f1": m.value}
    for m in val_f1
])

df_epoch = (
    train_loss_df
    .merge(val_loss_df, on="epoch", how="outer")
    .merge(train_f1_df, on="epoch", how="outer")
    .merge(val_f1_df, on="epoch", how="outer")
    .sort_values("epoch")
)

df_epoch.to_csv("outputs/best_model_curves.csv", index=False)

## Validation

### Load best model

In [ ]:
model = mlflow.pytorch.load_model(
    f"runs:/{best_run_id}/model",
    map_location=torch.device(ic.get_device()),
)

### Export thresholds

In [ ]:
pd.DataFrame(model.thresholds).to_csv("outputs/thresholds.csv", index=False)

### Export train and val predictions

In [ ]:
train, val = train_test_split(trainval,test_size=0.18,stratify=trainval["strat"],random_state=11)

os.makedirs("outputs/train", exist_ok=True)
train = train.drop("strat", axis=1)
proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=train, train_mode=False)))
proba.to_csv("outputs/train/train_proba.csv", index=False)
trainout = model.get_val_data(dataset=ic.FldDataset(data=train, train_mode=False))
trainout["predictions_revue"].to_csv("outputs/train/train_prediction_revue.csv", index=False)
trainout["classification_report"].to_csv("outputs/train/train_classification_report.csv", index=False)

os.makedirs("outputs/val", exist_ok=True)
val = val.drop("strat", axis=1)
proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=val, train_mode=False)))
proba.to_csv("outputs/val/val_proba.csv", index=False)
valout = model.get_val_data(dataset=ic.FldDataset(data=val, train_mode=False))
valout["predictions_revue"].to_csv("outputs/val/val_prediction_revue.csv", index=False)
valout["classification_report"].to_csv("outputs/val/val_classification_report.csv", index=False)

### Export test predictions

In [ ]:
os.makedirs("outputs/test", exist_ok=True)
proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=test, train_mode=False)))
proba.to_csv("outputs/test/test_proba.csv", index=False)
testout = model.get_val_data(dataset=ic.FldDataset(data=test, train_mode=False))
testout["predictions_revue"].to_csv("outputs/test/test_prediction_revue.csv", index=False)
testout["classification_report"].to_csv("outputs/test/test_classification_report.csv", index=False)

### Export holdout predictions

In [ ]:
os.makedirs("outputs/holdout", exist_ok=True)
proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=tab_holdout, train_mode=False)))
proba.to_csv("outputs/holdout/holdout_proba.csv", index=False)
holdoutout = model.get_val_data(dataset=ic.FldDataset(data=tab_holdout, train_mode=False))
holdoutout["predictions_revue"].to_csv("outputs/holdout/holdout_prediction_revue.csv", index=False)
holdoutout["classification_report"].to_csv("outputs/holdout/holdout_classification_report.csv", index=False)

### Export external predictions

In [ ]:
os.makedirs("outputs/external", exist_ok=True)
proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=tab_external, train_mode=False)))
proba.to_csv("outputs/external/external_proba.csv", index=False)
externalout = model.get_val_data(dataset=ic.FldDataset(data=tab_external, train_mode=False))
externalout["predictions_revue"].to_csv("outputs/external/external_prediction_revue.csv", index=False)
externalout["classification_report"].to_csv("outputs/external/external_classification_report.csv", index=False)

### Export holdout/external predictions per site

In [ ]:
sites = ["carpathians", "danube", "dovre", "french_alps", "sierra_nevada", "stubai_valley", "vinschgau"]

for site in sites:
    out_dir = f"outputs/sites/{site}"
    os.makedirs(out_dir, exist_ok=True)

    if site in sites_trainvaltest:
        subset = tab_holdout[tab_holdout["site"] == site].reset_index(drop=True)
    else:
        subset = tab_external[tab_external["site"] == site].reset_index(drop=True)
    
    dataset = ic.FldDataset(data=subset, train_mode=False)

    # probabilités
    proba = pd.DataFrame(model.predict_propabilities(dataset=dataset))
    proba.to_csv(f"{out_dir}/{site}_proba.csv", index=False)

    # sortie modèle
    challengeout = model.get_val_data(dataset=dataset)

    challengeout["predictions_revue"].to_csv(
        f"{out_dir}/{site}_prediction_revue.csv",
        index=False
    )

    challengeout["classification_report"].to_csv(
        f"{out_dir}/{site}_classification_report.csv",
        index=False
    )

### Zip outputs

In [ ]:
subprocess.run(["zip", "-rq", "outputs.zip", "outputs"], check=True)